In [56]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import os
from shapely.geometry import LineString, Point

In [46]:
import pandas as pd
import os

root_path = os.path.dirname(os.path.dirname(os.getcwd()))
DATA_PATH = "/GPX_RUNNING/data/input"

# Create columns correct names
CORRECT_COLS = [
    "type","drop","drop","drop",
    "timestamp","drop","drop","latitude",
    "drop","drop","longitude",
    "drop","drop","distance","drop",
    "drop","speed","drop","drop",
    "cadence", "drop", "drop", "drop", "drop", "drop"
]

def load_data(path: str):
    df = pd.DataFrame()
    files_read = 0
    for file in os.listdir(path):
        df = pd.concat([df, pd.read_csv(path + "/" + file)])
        files_read += 1
        break
    print(f"Number of files read are {files_read}")
    return df


raw_csv = load_data(root_path + DATA_PATH)

Number of files read are 1


In [47]:
def preprocess_cols(df):
    print(df.columns)        
    df = df[df.columns[0:25]].copy()
    # Create the proper column headers
    df = df.set_axis(CORRECT_COLS, axis=1)
    # Drop unused columns
    df = df.drop(columns="drop")
    df = df.dropna(axis=0)
    df = df[df['type'] == 'Data'].copy()
    print(df.columns)
    return df

In [48]:
df =preprocess_cols(raw_csv)

Index(['Type', 'Local Number', 'Message', 'Field 1', 'Value 1', 'Units 1',
       'Field 2', 'Value 2', 'Units 2', 'Field 3', 'Value 3', 'Units 3',
       'Field 4', 'Value 4', 'Units 4', 'Field 5', 'Value 5', 'Units 5',
       'Field 6', 'Value 6', 'Units 6', 'Field 7', 'Value 7', 'Units 7',
       'Field 8', 'Value 8', 'Units 8', 'Field 9', 'Value 9', 'Units 9',
       'Field 10', 'Value 10', 'Units 10', 'Field 11', 'Value 11', 'Units 11',
       'Field 12', 'Value 12', 'Units 12', 'Field 13', 'Value 13', 'Units 13',
       'Field 14', 'Value 14', 'Units 14', 'Field 15', 'Value 15', 'Units 15',
       'Field 16', 'Value 16', 'Units 16', 'Field 17', 'Value 17', 'Units 17',
       'Field 18', 'Value 18', 'Units 18', 'Field 19', 'Value 19', 'Units 19',
       'Field 20', 'Value 20', 'Units 20', 'Field 21', 'Value 21', 'Units 21',
       'Field 22', 'Value 22', 'Units 22', 'Unnamed: 69'],
      dtype='object')
Index(['type', 'timestamp', 'latitude', 'longitude', 'distance', 'speed',
    

In [49]:
df

,type,timestamp,latitude,longitude,distance,speed,cadence
5,Data,1118438300,0.000000e+00,6.100000e+02,0.000000e+00,652.000,60.000
9,Data,1118438300,3.964297e+09,1.000000e+00,2.148000e+03,3.400,0.000
10,Data,1118438300,1.000000e+00,2.148000e+03,3.500000e+00,1.000,0.000
11,Data,1118438300,3.964297e+09,1.000000e+00,2.148000e+03,3.400,2.000
17,Data,1118438300,4.911644e+08,-8.811374e+08,1.110000e+00,0.000,85.000
...,...,...,...,...,...,...,...
372,Data,1118441330,1.118438e+09,4.911644e+08,-8.811374e+08,2725.551,2725.551
374,Data,1118441330,1.000000e+05,4.001080e+05,1.118439e+09,0.000,1.000
375,Data,1118441330,1.609000e+05,6.536310e+05,1.118439e+09,0.000,1.000
376,Data,1118441330,5.000000e+05,2.102374e+06,1.118438e+09,0.000,1.000


In [50]:
def point_converter(meters_coords):
    # Create function to convert EPSG 3857 (meters) to EPSG 4326 (coordinates)
    boundary = meters_coords / ((2**32 )/ 360)
    return boundary

In [51]:
 # Convert the longitude and latitude to EPSG 4326
df["latitude"] = df["latitude"].apply(point_converter)
df["longitude"] = df["longitude"].apply(point_converter)

In [52]:
df.head()

,type,timestamp,latitude,longitude,distance,speed,cadence
5,Data,1118438300,0.000000e+00,5.112961e-05,0.00,652.0,60.0
9,Data,1118438300,3.322835e+02,8.381903e-08,2148.00,3.4,0.0
10,Data,1118438300,8.381903e-08,1.800433e-04,3.50,1.0,0.0
11,Data,1118438300,3.322835e+02,8.381903e-08,2148.00,3.4,2.0
17,Data,1118438300,4.116892e+01,-7.385608e+01,1.11,0.0,85.0


In [53]:
# !pip3 install openpyxl

In [54]:
df.to_excel("./test.xlsx")

In [55]:
df = df.dropna(subset=['latitude', 'longitude'])

# Filter out invalid coordinates (very large or very small values)
df = df[
    (df['latitude'] > -90) & (df['latitude'] < 90) &
    (df['longitude'] > -180) & (df['longitude'] < 180)
]

In [57]:
print("🔄 Creating geometry points...")
df['geometry'] = df.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)

# Create GeoDataFrame with proper CRS
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

🔄 Creating geometry points...


In [58]:
df

,type,timestamp,latitude,longitude,distance,speed,cadence,geometry
5,Data,1118438300,0.000000e+00,0.000051,0.000000e+00,652.000,60.0,POINT (5.112960934638977e-5 0)
10,Data,1118438300,8.381903e-08,0.000180,3.500000e+00,1.000,0.0,POINT (0.0001800432801247 8.381903171539307e-8)
17,Data,1118438300,4.116892e+01,-73.856081,1.110000e+00,0.000,85.0,POINT (-73.85608074255288 41.16892425343394)
20,Data,1118438306,4.116883e+01,-73.856132,1.331000e+01,2.153,0.0,POINT (-73.85613203980029 41.168831046670675)
21,Data,1118438314,4.116865e+01,-73.856056,3.429000e+01,2.666,83.0,POINT (-73.85605601593852 41.16865184158087)
...,...,...,...,...,...,...,...,...
366,Data,1118441026,8.381903e-08,0.000180,3.500000e+00,1.000,0.0,POINT (0.0001800432801247 8.381903171539307e-8)
374,Data,1118441330,8.381903e-03,0.033537,1.118439e+09,0.000,1.0,POINT (0.0335366651415825 0.0083819031715393)
375,Data,1118441330,1.348648e-02,0.054787,1.118439e+09,0.000,1.0,POINT (0.0547867175191641 0.0134864822030067)
376,Data,1118441330,4.190952e-02,0.176219,1.118438e+09,0.000,1.0,POINT (0.1762189529836178 0.0419095158576965)


In [62]:
# !pip3 install folium
!pip3 install geojson

In [ ]:
import folium
from folium.plugins import Search

import pandas as pd
import geopandas as gpd
import geopandas as gpd, folium, branca

from geojson import Point, Feature, FeatureCollection, dump
import json

import numpy as np
import matplotlib.pyplot as plt
import os 

from shapely.geometry import LineString, MultiLineString


def map_create():
    # Copy the data and use the copied version
    gdf = df.copy()

    # Convert the geometry column into a GeoJson to be usable for folium
    s = gdf["geometry"].to_json()
    s = json.loads(s)

   
    # open a new geojson file and write the current one into it
    with open('data/myfile.geojson', 'w') as f:
        dump(s, f)

    # read in the geojson file
    s2 = gpd.read_file('data/myfile.geojson', driver='GeoJSON')

    
    # use only certain column from the data file
    gdf2 = gdf[["geometry"]].copy()

    # Create map with initial location
    map1 = folium.Map(location=[43.7696, 11.2558], zoom_start=12)

    # Use the geojson to plot the runs w/ speed as the color and add to map
    speed_geo = folium.GeoJson(gdf2,
                            name='Track',
                            ).add_to(map1)
   
    # Add a LayerControl
    folium.LayerControl().add_to(map1)
   

   

    map1.save('data/index_speed.html')
        

In [73]:
!pip3 install geopandas

In [ ]:
import geopandas as gpd
# read the CSV file into a GeoDataFrame
gdf = gpd.read_file('/Users/vinny/Documents/GitHub/gpx_running/data/input/F1B90916.FIT.records.csv')

# convert the GeoDataFrame to a geojson object
# geo_json = gdf.to_json()

# however if the objects become very big, store the GeoDataFrame to a .geojson file
gdf.to_file('/Users/vinny/Documents/GitHub/gpx_running/data/input/test', driver='GeoJSON')

AttributeError: 'DataFrame' object has no attribute 'to_file'

In [66]:
map_create()

OverflowError: Maximum recursion level reached